# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mrittiga/flyrank-ml-internship/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
My provisional lane is Lane 2: Refresh / Content Opportunity Scoring.

I chose this lane because the starter project already demonstrates how search and content signals can be used to rank pages for human review. I want to investigate whether a transparent, evidence-based ranking can help a content team decide which pages deserve attention first, instead of relying only on fixed rules or intuition. I will start with the anonymized starter dataset and may refine the question after exploring the larger internship dataset.


## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
**Research question:** Can observable search-performance and content signals identify pages that should be prioritized for a human content-refresh review?

**Unit of analysis:** One content page per row in the starter dataset, using the supplied anonymized content-level snapshot.

**Decision to improve:** Which eligible pages should a content team review first when time and resources are limited?

**Output:** A ranked review queue with a score, supporting reason codes, and a suggested next action such as refresh, investigate, protect, or monitor.

**Action:** A human reviewer checks the evidence and page context before deciding whether to update the content, investigate a CTR issue, or take no action.

**Cost of a wrong recommendation:** A false positive wastes editorial time and may lead to an unnecessary change. A false negative may leave a genuine opportunity or sustained decline unexamined. An inappropriate refresh could also harm useful content.

**Why data or ML may help:** Multiple signals, including impressions, CTR, average position, content age, and engagement, may provide a more consistent way to prioritize review than a single fixed rule. A simple baseline will be compared with more complex methods before deciding whether ML adds value.


## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

In [5]:

import os
import subprocess
from pathlib import Path

repo_path = Path("/content/flyrank-ml-internship-starter")
repo_url = "https://github.com/flyrank-bih/flyrank-ml-internship-starter.git"

# Clone the repository if it isn't already present
if not repo_path.exists():
    subprocess.run(
        ["git", "clone", repo_url, str(repo_path)],
        check=True
    )

# Move into the repository
os.chdir(repo_path)

# Check whether the dataset exists
csv_path = repo_path / "data" / "raw" / "content_refresh_anonymized.csv"
print("Current directory:", os.getcwd())
print("CSV exists:", csv_path.exists())

if csv_path.exists():
    import pandas as pd
    df = pd.read_csv(csv_path)
    print("Dataset shape:", df.shape)
    display(df.head())
else:
    print("Dataset not found. Check the repository's data/raw folder.")


Current directory: /content/flyrank-ml-internship-starter
CSV exists: True
Dataset shape: (30000, 44)


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


In [2]:

import os
from pathlib import Path

print("Current directory:", os.getcwd())
print("CSV exists here:", Path("data/raw/content_refresh_anonymized.csv").exists())

# Check the current directory and common Colab locations
for folder in [Path("."), Path("/content"), Path("/content/flyrank-ml-internship-starter")]:
    csv_path = folder / "data/raw/content_refresh_anonymized.csv"
    print(csv_path, "->", csv_path.exists())


Current directory: /content
CSV exists here: False
data/raw/content_refresh_anonymized.csv -> False
/content/data/raw/content_refresh_anonymized.csv -> False
/content/flyrank-ml-internship-starter/data/raw/content_refresh_anonymized.csv -> False


In [6]:

# Step 2: Quick look at the dataset

print("Total rows:", len(df))
print("Total columns:", len(df.columns))

print("\nAvailable columns:")
print(df.columns.tolist())

# 1. Trend distribution
if "trend_direction" in df.columns:
    print("\nTrend distribution:")
    print(df["trend_direction"].value_counts())

# 2. Search volume vs. impressions
if {"search_volume", "impressions_90d"}.issubset(df.columns):
    correlation = df["search_volume"].corr(df["impressions_90d"])
    print("\nSearch volume vs. impressions correlation:", round(correlation, 3))

# 3. Median content length by trend
word_col = next(
    (c for c in ["word_count", "content_word_count"] if c in df.columns),
    None
)

if word_col and "trend_direction" in df.columns:
    print("\nMedian word count by trend:")
    print(df.groupby("trend_direction")[word_col].median())


Total rows: 30000
Total columns: 44

Available columns:
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']

Trend distribution:
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64

Search volume vs. impressions

In [ ]:
### Initial Data Evidence

The starter dataset contains **[insert row count] pages** across **[insert column count] columns**.

- **Trend patterns:** The dataset shows [insert the trend categories and counts], providing a basis for investigating which pages may need attention.
- **Search signals:** The correlation between search volume and impressions over the last 90 days is **[insert result]**. This helps assess whether these two signals provide similar or different information.
- **Content characteristics:** Median word counts vary across trend categories, which can be explored as a possible signal for prioritizing content reviews.

These are initial observations, not evidence that refreshing a page will cause its search performance to improve. Further validation is needed before recommending changes.

## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

### What I can and cannot claim

I can report observed associations in the anonymized starter dataset and compare ranking methods against a clearly defined target and evaluation metric.

I cannot claim that a correlation proves causation, that Google's algorithm uses a particular signal because of my findings, or that refreshing a recommended page will necessarily increase traffic.

The starter model predicts a proxy label based on the observed `trend_direction` value. This is not the same as predicting a future decline or proving that an intervention will cause recovery.

For later experiments, I will define the decision time and target window, check for data leakage, use appropriate validation such as client-holdout or time-based validation, and report limitations. Any recommendations will support human review rather than automatically trigger content changes.



## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.